# Python Script Node

This example notebook demonstrates how to create and use Python script nodes. This includes:

- Defining and running Python script nodes
- Debugging Python script nodes
- Dataflow: Receiving input from other skills and returning result values

**Prerequisites**

Python script nodes share many concepts with skills and skill task nodes. It is thus recommended to work through the corresponding example notebooks first:

- [Interacting with the world](003_world.ipynb)
- [Parameterizing skill instances](004_skills.ipynb)
- [Handling data flow](009_dataflow.ipynb)

**More examples**

More examples for Python script nodes can be found in the [frontend guide](https://flowstate.intrinsic.ai/docs/guides/develop_a_process/add_nodes/add_a_python_script/) for adding Python script nodes.



<div class="alert alert-info">

**Important**

This notebook requires a running Flowstate solution to connect to. To start a solution:

1. Navigate to [flowstate.intrinsic.ai](https://flowstate.intrinsic.ai/) and sign in
   using your registered Flowstate account.

1. Do **one** of the following:
    - Create a new solution:
        1. Click "Create new solution" and choose "From an example".
        1. Select `pick_and_place:pick_and_place_module2`
        1. Click "Create".
    - Or open an existing solution that was created from the `pick_and_place:pick_and_place_module2` example:
        1. Hover over the solution in the list.
        1. Click "Open solution" or "Start solution".

1. Recommended: Keep the browser tab with the Flowstate solution editor open to watch the effect of notebook actions such as running a skill. You can simultaneously interact with the solution through the web UI and the notebook.

</div>

## Create a Python script node

First, we connect to the solution and define some convenience shortcuts.

In [ ]:
from intrinsic.skills.python import basic_compute_context
from intrinsic.solutions import behavior_tree as bt
from intrinsic.solutions import deployments
from intrinsic.solutions import execution
from intrinsic.solutions import proto_building as pb
from intrinsic.solutions import worlds

solution = deployments.connect_to_selected_solution()

executive = solution.executive
skills = solution.skills
proto_builder = solution.proto_builder
world = solution.world
resources = solution.resources

capture_images = skills.ai.intrinsic.capture_images
move_robot = skills.ai.intrinsic.move_robot
plan_grasp = skills.ai.intrinsic.plan_grasp
estimate_pose = skills.ai.intrinsic.estimate_pose

Python script nodes—like all other node types—are available through the `behavior_tree` (=`bt`) module. You can create an instance and run it directly with `executive.run()`.

In [ ]:
hello_world = bt.PythonScript(
    function_body="print('Hello world!')",
)

executive.run(hello_world)

To include Python script nodes in a behavior tree, you must wrap them in a `Task` node which also allows you to assign a helpful name.

In [ ]:
# Directly run single node.
executive.run(bt.Task(name="My first script", action=hello_world))

# Add node to a tree and run the tree.
tree = bt.BehaviorTree(root=bt.Task(name="My first script", action=hello_world))
executive.run(tree)

## Inspect the execution output

In the frontend, you can view the output of a Python script node in the sequence list.

<img src="images/pynode_stdout.png" 
     style="border: 1px solid black; width: 75%"
     alt="View stdout of Python script node in frontend" />
 
After running the above cells:

1. You should see a **Current Run** with executive state **Succeeded**.
2. Click on the **Load sequence list** button or on the reload button next to the sequence list.
3. For the **My first script** node in the sequence list, click on the button for viewing printed messages.

You should see the log output "Hello world!" from our script node, as shown in the screenshot.

## Inspect execution errors

Create a node that will produce a Python error and run it. In a Jupyter notebook this will automatically print a pretty HTML error report with the Python error caused by the invalid Python code. In regular Python scripts, you can handle or print the `ExecutionFailedError` as you would with other Python errors.

In [ ]:
failing_script = bt.PythonScript(
    proto_builder.create_signature_with_args(),
    # Causes: "TypeError: unsupported operand type(s) for +: 'int' and 'str'"
    function_body="print(5 + 'foo')",
)

try:
    executive.run(bt.Task(name="Failing script", action=failing_script))
except execution.ExecutionFailedError:
    # No additional error handling, the automatically printed HTML
    # report is sufficient here.
    pass

As an alternative to inspecting and handling the Python error here in the notebook or in a regular Python script, you can also see the error in the sequence list in the frontend.

<img src="images/pynode_error.png" 
     style="border: 1px solid black; width: 75%"
     alt="View error of Python script node in frontend" />

## Inputs and outputs

Python script nodes can have inputs and outputs. This works very similarly to skills, in that the output and input of each node is defined by a *parameter* and *return value* Protobuf message. These messages can be created with the help of `solution.proto_builder.create_signature_with_args()` which creates a so-called `SignatureWithArgs` object that can be passed to the `PythonScript` constructor. The `SignatureWithArgs` object encapsulates the script node's "signature" in the form of definitions of the aforementioned parameter and return value messages as well as concrete "args" to be passed to the individual parameters. The arguments work analogously to skill constructors. They can be set to fixed values or they can be dynamically assigned from values on the blackboard at runtime (e.g., results from other skills).

Let's take a look at a first example. This does not have any dynamic assignments from the blackboard - we only pass fixed values to the `arg` parameter of `FieldSpec`.

In [ ]:
add_script = bt.Task(
    name="Add 42 and 24",
    action=bt.PythonScript(
        proto_builder.create_signature_with_args(
            # In the field specs of the parameter message spec (and only there)
            # you can use 'arg' to define which arguments are passed.
            parameters=pb.MessageSpec(fields=[
                pb.FieldSpec(type="int64", name="a", number=1, arg=42),
                pb.FieldSpec(type="int64", name="b", number=2, arg=24)]),
            return_value=pb.MessageSpec(fields=[
                pb.FieldSpec(type="int64", name="sum", number=1)
            ])),
        # Optional. If omitted, a key will be auto-generated.
        return_value_key="my_custom_return_value_key",
        function_body="return node_pb2.ReturnValue(sum=params.a + params.b)",
    )
)

We can run this script node and then inspect the return value message on the blackboard.

In [ ]:
executive.run(add_script)

result_any = executive.operation.blackboard.get_value_any(add_script.result)
# ... or equivalently, if you have manually specified a return value key ...
result_any = executive.operation.blackboard.get_value_any("my_custom_return_value_key")

result = add_script.result.value_type()
result_any.Unpack(result)

print(f"Result message:\n{result}")

The `proto_builder` class internally creates an actual proto file from the given message specs and runs `protoc` on it. It uses fixed names for the generated proto messages and the proto package and filename in which they are placed. This allows you to use the proto classes `node_pb2.Params` and `node_pb2.ReturnValue` in the function body. For more details about the available names in the function body, see the [next section](#function-body).

The `MessageSpec` and `FieldSpec` classes offer a structured way to define the two messages required for a signature, and they additionally provide convenience features such as specifying an `arg` inline with its corresponding message field. Except for `arg`, the other parameters of the `FieldSpec` class directly map to elements of the [proto syntax](https://protobuf.dev/programming-guides/proto3/#simple). Just as in `.proto` files, you must not only specify a *name* for each field but also a field [*number*](https://protobuf.dev/programming-guides/proto3/#assigning) which should be unique and never be reused when you add or remove fields.

The field *type* can be either a [Protobuf scalar type](https://protobuf.dev/programming-guides/proto3/#scalar) (`string`, `int64`, `bool`, `double`, ...) or one of the "well-known" types supported by the proto builder. These include several [Protobuf well-known types](https://protobuf.dev/reference/protobuf/google.protobuf/) as well as the most important protos from the Intrinsic SDK. You can get an up-to-date list of supported types with `proto_builder.get_well_known_types()`.

In [ ]:
proto_builder.get_well_known_types()

## Add to the function body and preview the executed code <a id="function-body"></a>

The "function body" provided to the constructor of a `PythonScript` needs to be valid Python code. It will be executed in a remote environment when the behavior tree is run. In this remote environment the code is inserted into a template script as the body of a function, so that constructs like a `context` object or return statements can be used in the user-provided code snippet without additional boilerplate. It is possible to preview an equivalent of the final, expanded script that will be executed in the remote environment to get a better idea of, e.g., which variables are available.

Let's start with a slightly more complex script node which has a few interesting inputs and outputs and which uses the `context` object to create a frame in the world.

In [ ]:
script_with_inouts = bt.Task(
    name="Script with inouts and world access",
    action=bt.PythonScript(
        proto_builder.create_signature_with_args(
            parameters=pb.MessageSpec(fields=[
                pb.FieldSpec(type="string", name="name_in",
                             number=1, arg="my_frame"),
                pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                             name="parent_in", number=2, arg=world.building_block0),
            ]),
            return_value=pb.MessageSpec(fields=[
                pb.FieldSpec(type="int64", name="rand_out", number=1),
            ]),
        ),
        function_body="""
            import random

            world = context.object_world
            parent_node = world.get_transform_node(params.parent_in)
            world.create_frame(params.name_in, parent_node)

            return node_pb2.ReturnValue(rand_out=random.randint(1, 10))
        """,
    )
)

executive.run(script_with_inouts, start_from_world_state=worlds.EditWorldId.INITIAL)

It may not be immediately obvious that this code works, but we can confirm it by previewing the executed code using `solution.code_execution_info.preview_executed_code()`. 

In [ ]:

print(solution.code_execution_info.preview_executed_code(script_with_inouts))

<div class="alert alert-info">

**Tip**

For technical reasons `solution.code_execution_info.preview_executed_code()` requires a fully constructed `PythonScript` instance or a `Task` instance wrapping a `PythonScript`. The function body of the `PythonScript` does not need to be valid to get a preview. If you don't know the function body yet, you can use a placeholder such as `function_body="pass"` when you construct the `PythonScript`.

</div>

In the previewed script you can see how the user-provided code is interpreted as the body of a function called `compute`. The types of the parameter and return value message are provided through the imported proto module `node_pb2`. The input parameters are available as a proto message `params: node_pb2.Params` and there is an additional `context: basic_compute_context.BasicComputeContext` which provides, e.g., access to the world. The function is expected to return an instance of `node_pb2.ReturnValue`.

The output of `preview_executed_code()` is generally similar, but specific details vary depending on whether the signature of the Python script node defines parameter and/or return value messages.

## Advanced: Write a function body with language support

Writing the function body as a string literal can be inconvenient and error-prone because features such as syntax highlighting, auto-completion, and auto-formatting are unavailable. Depending on your development environment you can use the following approach to get *basic* language support. You can write an actual Python function in your development environment and then pass this function's body to be used in the Python script node. To achieve that you can use the helper `bt.get_function_body_as_str()` (which uses Python's `inspect.getsource()` under the hood). The language support that is available with this approach covers general Python as well as modules from the Intrinsic SDK. It does not cover the proto classes for the dynamically created signature (`node_pb2.Params` and `node_pb2.ReturnValue`).

In [ ]:

# Python function that contains the code which is passed to the Python
# script node function body.
def script_with_inouts_fn(
    # Types unknown in the local environment must be quoted so that this
    # function definition can be executed.
    params: 'node_pb2.Params',
    context: basic_compute_context.BasicComputeContext
) -> 'node_pb2.ReturnValue':
    # Import everything we need, we have no access to global imports
    # from the local environment here.
    import random
    
    world = context.object_world
    parent_node = world.get_transform_node(params.parent_in)
    world.create_frame(params.name_in, parent_node)

    # Unknown names in the function's body don't bother the Python
    # interpreter as long as we don't call this function.
    return node_pb2.ReturnValue(rand_out=random.randint(1, 10))


script_with_inouts = bt.Task(
    name="Script with inouts and world access",
    action=bt.PythonScript(
        proto_builder.create_signature_with_args(
            parameters=pb.MessageSpec(fields=[
                pb.FieldSpec(type="string", name="name_in",
                             number=1, arg="my_frame"),
                pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                             name="parent_in", number=2, arg=world.building_block0),
            ]),
            return_value=pb.MessageSpec(fields=[
                pb.FieldSpec(type="int64", name="rand_out", number=1),
            ]),
        ),
        function_body=bt.get_function_body_as_str(script_with_inouts_fn),
    )
)

executive.run(script_with_inouts, start_from_world_state=worlds.EditWorldId.INITIAL)


<div class="alert alert-warning">

**Warning**

Using `get_function_body_as_str()` is an advanced technique which requires Python experience. 

- The local function is just a helper for generating the function body of the `PythonScript`.
- The local function cannot be executed in the local Python environment.
- The local function's body may not access any variables from the local environment, such as global imports.
- The local function's body may access any variables from the remote environment (see how to [preview the executed code](#function-body)).
- The local function's header (name and signature) is completely ignored. You could use `def foo():` without any parameters and the code above would still work in the same way.

</div>

## Examples: Modify data of motion skills

The main value of Python script nodes lies in connecting them with inputs and outputs of other nodes and process the data that flows between them. Defining input and output data works just like it does for skill task nodes. You can use the `.result` property of a script task node to access the result value and you can pass the `.result` of other nodes to the arguments of a Python script node to create dynamic assignments which will be evaluated when the process is run.

To showcase more examples we first create a few basic motion skills.

In [ ]:
move_home = bt.Task(
    name="Move home",
    action=move_robot(
         motion_segments=[
             move_robot.intrinsic_proto.skills.MotionSegment(
                 joint_position=world.robot.joint_configurations.home,
                 motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,

             )
         ],
        arm_part=world.robot,
    )
)

move_view_pose_left = bt.Task(
    name="Move to left view pose",
    action=move_robot(
         motion_segments=[
             move_robot.intrinsic_proto.skills.MotionSegment(
                 joint_position=world.robot.joint_configurations.view_pose_left,
                 motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
             )
         ],
        arm_part=world.robot,
    )
)

### Basic dataflow

In the following example we pass the result pose from the `estimate_pose` skill to a Python script node. The script node generates an "approach" pose above the result pose from the `estimate_pose` skill and returns it. We then move the robot to this approach pose by passing the script node result to the target pose of the `move_robot` skill.

In [ ]:
capture_image = bt.Task(
    name="Capture image",
    action=capture_images(camera=resources.wrist_camera),
)

estimate_block = bt.Task(
    name="Estimate block",
    action=estimate_pose(
        capture_data=capture_image.result.capture_data,
        pose_estimator=solution.pose_estimators.building_block_ml_estimator,
    )
)


def generate_approach_pose_fn(
    params: 'node_pb2.Params',
    context: basic_compute_context.BasicComputeContext
) -> 'node_pb2.ReturnValue':
    from intrinsic.math.python import data_types
    from intrinsic.math.python import proto_conversion

    root_t_object = proto_conversion.pose_from_proto(params.root_t_object)

    # Ignore rotation of estimated pose. Move to a position above the estimated
    # pose with a fixed rotation (so that the gripper points downwards).
    root_t_approach = data_types.Pose3(
        rotation=data_types.Rotation3.from_euler_angles(
            rpy_degrees=[180, 0, 180]),
        translation=root_t_object.translation + (0, 0, params.distance),
    )

    return node_pb2.ReturnValue(
        root_t_approach=proto_conversion.pose_to_proto(root_t_approach))


generate_approach_pose = bt.Task(
    name="Generate approach pose",
    action=bt.PythonScript(
        proto_builder.create_signature_with_args(
            parameters=pb.MessageSpec(fields=[
                # Estimated pose, assigned from result of "estimate_pose" skill.
                pb.FieldSpec(type="intrinsic_proto.Pose", name="root_t_object", number=1,
                             arg=estimate_block.result.root_ts_target[0]),
                # Distance of approach pose above estimated position, set to a fixed value.
                pb.FieldSpec(type="double", name="distance",
                             number=2, arg=0.1),
            ]),
            return_value=pb.MessageSpec(fields=[
                pb.FieldSpec(type="intrinsic_proto.Pose",
                             name="root_t_approach", number=1),
            ])),
        function_body=bt.get_function_body_as_str(generate_approach_pose_fn),
    )
)

move_approach = bt.Task(
    name="Move to approach",
    action=move_robot(
        motion_segments=[
            move_robot.intrinsic_proto.skills.MotionSegment(
                cartesian_pose=move_robot.intrinsic_proto.motion_planning.v1.PoseEquality(
                    moving_frame=world.picobot_gripper.tool_frame,
                    target_frame=world.root,
                    # Assign result of script node.
                    target_frame_offset=generate_approach_pose.result.root_t_approach,
                ),
                motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
            )
        ],
        arm_part=world.robot,
    )
)

executive.run(
    bt.BehaviorTree(
        name="Move to approach pose over block",
        root=bt.Sequence([
            move_view_pose_left,
            capture_image,
            estimate_block,
            generate_approach_pose,
            move_approach,
        ]),
    ),
    start_from_world_state=worlds.EditWorldId.INITIAL,
)

### Dataflow with loop

The following example demonstrates how a Python script node can be used to generate data for a loop. We first run a Python script node which generates and returns a list of target poses. Then we move the robot to each of the target poses by running a loop over the list of poses and passing each pose as the target pose to the `move_robot` skill.

In [ ]:
def generate_poses_fn(
    params: 'node_pb2.Params',
    context: basic_compute_context.BasicComputeContext,
) -> 'node_pb2.ReturnValue':
    from intrinsic.math.python import data_types
    from intrinsic.math.python import proto_conversion

    world = context.object_world
    base = world.get_transform_node(params.base)
    root_t_base = world.get_transform(world.root, base)

    # Three hardcoded offsets for simplicity. These could also depend on, e.g.,
    # random numbers or input parameters.
    offsets = [
        [-0.1, 0, 0.1],
        [0.0, 0, 0.1],
        [0.1, 0, 0.1],
    ]

    # Ignore rotation of base pose. Move to a position above the base
    # pose with a fixed rotation (so that the gripper points downwards).
    root_ts_target = [
        data_types.Pose3(
            rotation=data_types.Rotation3.from_euler_angles(rpy_degrees=[180, 0, 180]),
            translation=root_t_base.translation + offset,
        )
        for offset in offsets
    ]

    return node_pb2.ReturnValue(root_ts_target=[
        proto_conversion.pose_to_proto(root_t_target)
        for root_t_target in root_ts_target
    ])


generate_poses = bt.Task(
    name="Generate offsets",
    action=bt.PythonScript(
        proto_builder.create_signature_with_args(
            parameters=pb.MessageSpec(fields=[
                # Frame above which to generate the target poses.
                pb.FieldSpec(type="intrinsic_proto.world.TransformNodeReference",
                             name="base", number=1, arg=world.building_block_tray)
            ]),
            return_value=pb.MessageSpec(fields=[
                pb.FieldSpec(type="intrinsic_proto.Pose", name="root_ts_target",
                             repeated=True, number=1)
            ])),
        function_body=bt.get_function_body_as_str(generate_poses_fn),
    )
)

loop = bt.Loop(name="Move to targets")

# Loop over the size of "root_ts_target" as returned by the script node.
loop.set_while_condition(bt.Blackboard(
    f"{loop.loop_counter} < size({generate_poses.result.root_ts_target.value_access_path()})"
))

move_approach = bt.Task(
    name="Move to single target",
    action=move_robot(
        motion_segments=[
            move_robot.intrinsic_proto.skills.MotionSegment(
                cartesian_pose=move_robot.intrinsic_proto.motion_planning.v1.PoseEquality(
                    moving_frame=world.picobot_gripper.tool_frame,
                    target_frame=world.root,
                    # Use "root_ts_target[i]" as target frame (with i being the loop counter).
                    target_frame_offset=generate_poses.result.root_ts_target[loop.loop_counter],
                ),
                motion_type=move_robot.intrinsic_proto.skills.MotionSegment.MotionType.ANY,
            )
        ],
        arm_part=world.robot
    )
)

loop.set_do_child(move_approach)

executive.run(
    bt.BehaviorTree(
        name="Move to generated poses",
        root=bt.Sequence([
            move_home,
            generate_poses,
            loop,
        ]),
    ),
    start_from_world_state=worlds.EditWorldId.INITIAL,
)